# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.01 · Cascada de etiquetado calibrada

Reproduce el patrón histórico Flash→Pro: calibra una primera pasada económica, etiqueta el corpus por lotes y dirige los casos riesgosos a un revisor más capaz.

La procedencia de `deepseek-v4-flash` y `deepseek-v4-pro` está documentada por el proveedor [1], al igual que sus precios por tokens y caché [2]. La selección dirigida pertenece a la familia de aprendizaje activo [3]. El acuerdo Flash–Pro calibra una regla operativa, pero no constituye *ground truth*; las tareas subjetivas conservan una instancia humana final independiente [4]. Los umbrales, el presupuesto, el control seguro y la precedencia son decisiones locales auditables.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Backend opcional Google Colab desde VS Code

Instale la extensión oficial **Google Colab** (`google.colab`), seleccione `Select Kernel > Colab`; esta campaña API funciona con runtime CPU. El notebook permanece local; Drive transporta solo versiones inmutables del bundle. Si la copia activa no coincide, la celda lee `bundle_releases/latest.json`, verifica todos sus SHA-256 y promueve automáticamente esa versión. Ejecute antes `02_00` directamente en Colab. Edite `COLAB_RUN_ID` para separar experimentos. La compatibilidad de `drive.mount()` desde VS Code requiere la extensión v0.2.1 o posterior [5]. La integridad del bundle se comprueba con SHA-256 [6]. No sincronice cachés de modelos ni escriba checkpoints directamente en Drive.

In [10]:
# Backend reproducible: local o Google Colab desde VS Code
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile

COLAB_NOTEBOOK_ID = "02_01"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = True
COLAB_AUTO_UPDATE_BUNDLE = True
COLAB_NOTEBOOK_BUILD_BUNDLE_ID = "b03ac0357ec959a4ba38869cfc1d0b311366105e7fbc05edb605e7fe8d23fd9b"  # Trazabilidad al generar el notebook
COLAB_EXPECTED_CORE_SHA256 = "44b088fe2eacbe6b4ab473f0c535bbde01517791b13a2287430a5717f809f6f1"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

# Los modelos configurados son públicos. Evita que huggingface_hub intente
# consultar el vault de secretos, que solo funciona desde la interfaz web de Colab.
if IN_COLAB:
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HOME"] = "/content/huggingface"

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

def _read_manifest(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _bundle_id_for_manifest(manifest):
    core = manifest["core"]
    inputs = manifest["inputs"]
    identity = {
        "schema_version": manifest["schema_version"],
        "taxonomy_contract": manifest["taxonomy_contract"],
        "taxonomy_version": manifest["taxonomy_version"],
        "core": {"name": core["name"], "sha256": core["sha256"]},
        "inputs": {
            key: {
                "archive": value["archive"],
                "archive_sha256": value["archive_sha256"],
                "source_sha256": value["source_sha256"],
            }
            for key, value in sorted(inputs.items())
        },
    }
    payload = json.dumps(identity, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def _bundle_specs(manifest):
    specs = [(manifest["core"]["name"], manifest["core"]["sha256"])]
    specs.extend(
        (entry["archive"], entry["archive_sha256"])
        for entry in manifest.get("inputs", {}).values()
    )
    for name, expected_sha256 in specs:
        if Path(name).name != name or not expected_sha256:
            raise ValueError(f"Entrada insegura o incompleta en bundle_manifest.json: {name!r}")
    return specs

def _bundle_is_current(bundle_dir, manifest_path, expected_bundle_id):
    if not manifest_path.is_file():
        return False
    try:
        manifest = _read_manifest(manifest_path)
        if manifest.get("bundle_id") != _bundle_id_for_manifest(manifest):
            return False
        if manifest["bundle_id"] != expected_bundle_id:
            return False
        if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
            return False
        return all(
            (bundle_dir / name).is_file() and _sha256(bundle_dir / name) == expected_sha256
            for name, expected_sha256 in _bundle_specs(manifest)
        )
    except (KeyError, TypeError, ValueError, json.JSONDecodeError):
        return False

def _activate_verified_drive_release(release_dir, bundle_dir, expected_bundle_id):
    release_manifest_path = release_dir / "bundle_manifest.json"
    if not _bundle_is_current(release_dir, release_manifest_path, expected_bundle_id):
        raise RuntimeError(
            "La versión esperada no está completa o no coincide con sus SHA-256: " + str(release_dir)
        )
    manifest = _read_manifest(release_manifest_path)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    # Todos los artefactos se validaron antes; el manifiesto activo se reemplaza al final.
    for name, _ in _bundle_specs(manifest):
        partial = bundle_dir / f".{name}.partial"
        shutil.copyfile(release_dir / name, partial)
        os.replace(partial, bundle_dir / name)
    partial_manifest = bundle_dir / ".bundle_manifest.json.partial"
    shutil.copyfile(release_manifest_path, partial_manifest)
    os.replace(partial_manifest, bundle_dir / "bundle_manifest.json")
    if not _bundle_is_current(bundle_dir, bundle_dir / "bundle_manifest.json", expected_bundle_id):
        raise RuntimeError("La activación desde bundle_releases no superó la verificación final")
    return manifest

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    RELEASES_DIR = DRIVE_ROOT / "bundle_releases"
    latest_pointer_path = RELEASES_DIR / "latest.json"
    latest_pointer = _read_manifest(latest_pointer_path) if latest_pointer_path.is_file() else {}
    latest_bundle_id = str(latest_pointer.get("bundle_id") or "")
    latest_matches_notebook = (
        len(latest_bundle_id) == 64
        and latest_bundle_id == COLAB_NOTEBOOK_BUILD_BUNDLE_ID
        and latest_pointer.get("core_sha256") == COLAB_EXPECTED_CORE_SHA256
    )
    if latest_matches_notebook:
        release_source = "latest_pointer"
        RELEASE_DIR = RELEASES_DIR / latest_bundle_id
        expected_manifest_sha256 = latest_pointer.get("manifest_sha256")
    else:
        # Un cuaderno reproducible puede activar su release inmutable exacto aunque
        # latest todavía apunte a otra versión; jamás mezcla código e inputs.
        release_source = "notebook_pinned_release"
        latest_bundle_id = COLAB_NOTEBOOK_BUILD_BUNDLE_ID
        RELEASE_DIR = RELEASES_DIR / latest_bundle_id
        pinned_manifest_path = RELEASE_DIR / "bundle_manifest.json"
        if not _bundle_is_current(RELEASE_DIR, pinned_manifest_path, latest_bundle_id):
            raise RuntimeError(
                "Drive no contiene ni latest compatible ni el release inmutable fijado por este "
                "cuaderno. Ejecute 02_00_preparacion_bundle_colab.ipynb y publique el bundle exacto."
            )
        pinned_manifest = _read_manifest(pinned_manifest_path)
        if pinned_manifest.get("core", {}).get("sha256") != COLAB_EXPECTED_CORE_SHA256:
            raise RuntimeError("El release fijado por el cuaderno contiene un core inesperado")
        expected_manifest_sha256 = _sha256(pinned_manifest_path)
    release_manifest_path = RELEASE_DIR / "bundle_manifest.json"
    if not release_manifest_path.is_file() or _sha256(release_manifest_path) != expected_manifest_sha256:
        raise RuntimeError("El manifiesto del release de Drive falta o no coincide con su referencia")
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    bundle_activated = False
    modules_loaded_before_update = any(
        name == "moderacion_peru" or name.startswith("moderacion_peru.") for name in sys.modules
    )
    if not _bundle_is_current(BUNDLE_DIR, manifest_path, latest_bundle_id):
        if not COLAB_AUTO_UPDATE_BUNDLE:
            raise RuntimeError("El bundle de Drive está desactualizado y COLAB_AUTO_UPDATE_BUNDLE=False")
        try:
            manifest = _activate_verified_drive_release(RELEASE_DIR, BUNDLE_DIR, latest_bundle_id)
            bundle_activated = True
        except Exception as exc:
            raise RuntimeError(
                "No fue posible activar la versión esperada desde Google Drive. Ejecute en Colab "
                "02_00_preparacion_bundle_colab.ipynb, confirme status=published_to_drive y "
                f"compruebe que exista {RELEASE_DIR}."
            ) from exc
    else:
        manifest = _read_manifest(manifest_path)

    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    if bundle_activated and modules_loaded_before_update:
        raise RuntimeError(
            "El bundle se actualizó y verificó en Drive, pero este kernel ya había importado una "
            "versión anterior de moderacion_peru. Reinicie completamente el kernel de Colab y vuelva "
            "a ejecutar el cuaderno desde la primera celda."
        )

    os.environ["MODPERU_ROOT"] = str(ROOT)
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_result('Bundle de Colab verificado', {
        'estado': 'activado_desde_drive' if bundle_activated else 'ya_estaba_actualizado',
        'bundle_id': manifest['bundle_id'],
        'bundle_del_notebook_al_generarse': COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
        'origen_del_release': release_source,
        'core_sha256': expected_core,
        'generado': manifest.get('generated_at'),
        'versión_inmutable_drive': RELEASE_DIR,
    }, tone='success')
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


raíz,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4
backend,local


## Configuración explícita y credencial

In [11]:
import os
from pathlib import Path
from moderacion_peru.providers import DeepSeekProvider

if globals().get('IN_COLAB') and not os.getenv('DEEPSEEK_API_KEY'):
    from google.colab import userdata

    try:
        os.environ['DEEPSEEK_API_KEY']=userdata.get('DEEPSEEK_API_KEY') or ''
    except Exception:
        pass

SOURCE=COLAB_CONTEXT.input('chunks_v2') if COLAB_CONTEXT else ROOT/'datos/processed/chunks_v2.jsonl'
CAMPAIGN_ROOT=COLAB_CONTEXT.scratch_output_dir if COLAB_CONTEXT else ROOT/'datos/etiquetado/cascada_deepseek_v4'
CAMPAIGN_ROOT.mkdir(parents=True,exist_ok=True)
HISTORICAL_CHUNKS=COLAB_CONTEXT.input('chunks_deepseek_historicos') if COLAB_CONTEXT else ROOT/'datos/processed/chunks_para_etiquetar.jsonl'
HISTORICAL_FLASH_SOURCES=(COLAB_CONTEXT.input('deepseek_flash_historico'),) if COLAB_CONTEXT else (ROOT/'datos/etiquetado/llm_api/deepseek-v4-flash_labeled_chunks_seed42.jsonl',)
HISTORICAL_PRO_SOURCES=(COLAB_CONTEXT.input('deepseek_pro_historico_principal'),COLAB_CONTEXT.input('deepseek_pro_historico_umbral'),COLAB_CONTEXT.input('deepseek_pro_historico_sospechosos')) if COLAB_CONTEXT else (ROOT/'datos/etiquetado/llm_api/deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.jsonl',ROOT/'datos/etiquetado/llm_api/deepseek-v4-pro_revision_umbral_recalibrado_t090_seed42.jsonl',ROOT/'datos/etiquetado/llm_api/deepseek-v4-pro_revision_sospechosos_gruesos_seed42.jsonl')
HISTORICAL_PROMPT_SHA256='52d4fec14ad433d35ec20de5f51a6954aad69dcedd1422059419dcecc2f9e778'
PRIMARY_PATH=CAMPAIGN_ROOT/'primary_flash.jsonl'
REVIEW_PATH=CAMPAIGN_ROOT/'review_pro.jsonl'
RECOVER_HISTORICAL=True  # Recupera solo coincidencias exactas 1:1; nunca transfiere segmentos distintos.
AUTO_PUBLISH_CHECKPOINTS=True  # En Colab publica TAR.GZ atómico al recuperar, periódicamente y al interrumpir.
DRIVE_CHECKPOINT_EVERY_BATCHES=10  # 10 ventanas × 160 chunks; cada grupo de 5 ya queda fsync local.
RUN_API_PREFLIGHT=True  # Consulta /models y /user/balance sin enviar textos ni consumir tokens de etiquetado.
RUN_CALIBRATION=True  # Primero: panel pareado Flash–Pro. Costo esperado muy bajo.
RUN_PRIMARY=True      # Después: primera pasada Flash.
RUN_DIRECTED_REVIEW=True  # Al final: Pro solo sobre la cola dirigida.
CALIBRATION_PANEL_SIZE=1000  # Aún breve; permite evaluar LI95%≈0.95 con potencia útil.
PRIMARY_LIMIT=None  # None para TODOS y solo los pendientes; use 20 únicamente para un smoke test. Nunca lo deje en blanco.
REVIEW_LIMIT=None   # Campaña: TODA y solo la cola dirigida pendiente.
PROCESSING_BATCH_SIZE=160  # 32 solicitudes concurrentes × 5 registros.
MAX_PRIMARY_COST_USD=60.0
MAX_REVIEW_COST_USD=25.0
BALANCE_REFRESH_SECONDS=60.0  # Consulta de saldo sin corpus durante la ejecución.
LOW_BALANCE_WARNING_USD=2.0
CACHE_ALERT_AFTER_REQUESTS=50
MIN_CACHE_HIT_RATE=0.50
SAFE_CONTROL_RATE=0.10

primary_provider=DeepSeekProvider(model='deepseek-v4-flash',max_workers=32,records_per_request=5,max_cost_usd=MAX_PRIMARY_COST_USD,label_source='deepseek_remote')
reviewer_provider=DeepSeekProvider(model='deepseek-v4-pro',max_workers=32,records_per_request=5,max_cost_usd=MAX_REVIEW_COST_USD,label_source='llm_remote_review')
primary_probe=primary_provider.probe(); reviewer_probe=reviewer_provider.probe()
expected_thinking={'type':'disabled'}
expected_response_format={'type':'json_object'}
expected_cache_usage_fields=['prompt_cache_hit_tokens','prompt_cache_miss_tokens']
if primary_probe['thinking'] != expected_thinking or reviewer_probe['thinking'] != expected_thinking:
    raise RuntimeError('02_01 exige DeepSeek V4 en modo non-thinking para Flash y Pro')
if primary_probe['response_format'] != expected_response_format or reviewer_probe['response_format'] != expected_response_format or primary_probe['output_contract']['root_key'] != 'annotations' or reviewer_probe['output_contract']['root_key'] != 'annotations':
    raise RuntimeError('02_01 exige JSON object con raíz annotations para Flash y Pro')
if primary_probe['context_cache']['mode'] != 'automatic_prefix' or reviewer_probe['context_cache']['mode'] != 'automatic_prefix':
    raise RuntimeError('02_01 exige caché de contexto automática con prefijo estable')
if primary_probe['context_cache']['verified_from_usage_fields'] != expected_cache_usage_fields or reviewer_probe['context_cache']['verified_from_usage_fields'] != expected_cache_usage_fields:
    raise RuntimeError('02_01 debe medir aciertos y fallos reales de caché en la respuesta de DeepSeek')
if not primary_probe['credential_configured']:
    show_callout('Falta credencial','Defina DEEPSEEK_API_KEY en el entorno o como secreto de Colab. El preflight no consume crédito.',tone='warning')
show_result('Primera pasada',primary_probe,tone='success')
show_result('Revisor dirigido',reviewer_probe,tone='success')
show_summary('Modo DeepSeek verificado',{'Flash':primary_probe['thinking'],'Pro':reviewer_probe['thinking'],'JSON_Flash':primary_probe['response_format'],'JSON_Pro':reviewer_probe['response_format'],'contrato_salida':primary_probe['output_contract'],'caché_Flash':primary_probe['context_cache'],'caché_Pro':reviewer_probe['context_cache']},tone='success')
if RUN_API_PREFLIGHT and primary_probe['credential_configured']:
    flash_connection=primary_provider.validate_connection(); pro_connection=reviewer_provider.validate_connection()
    if not flash_connection['model_available'] or not pro_connection['model_available']:
        raise RuntimeError('Flash o Pro no aparece disponible en el catálogo de DeepSeek')
    initial_balance=primary_provider.balance_summary()
    show_result('Credencial, modelos y saldo verificados; no se enviaron textos',{'Flash':flash_connection,'Pro':pro_connection,'saldo':initial_balance},tone='success' if initial_balance['is_available'] else 'warning')
show_summary('Rutas y activación',{'entrada':SOURCE,'campaña':CAMPAIGN_ROOT,'recuperación_histórica':RECOVER_HISTORICAL,'checkpoint_drive_automático':bool(COLAB_CONTEXT and AUTO_PUBLISH_CHECKPOINTS),'calibración':RUN_CALIBRATION,'primaria':RUN_PRIMARY,'revisión':RUN_DIRECTED_REVIEW},tone='neutral')

provider,deepseek_http
base_url,https://api.deepseek.com
model,deepseek-v4-flash
credential_configured,Sí
network_called,No
max_workers,32
records_per_request,5
max_cost_usd,60.0
label_source,deepseek_remote
operational_prompt_path,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\config\prompt_operacional_ollama_v2.md
operational_prompt_sha256,94fa2a479c1995aa2cd27d9d2767e9f5bee33e56129785b393f7224b12967838


provider,deepseek_http
base_url,https://api.deepseek.com
model,deepseek-v4-pro
credential_configured,Sí
network_called,No
max_workers,32
records_per_request,5
max_cost_usd,25.0
label_source,llm_remote_review
operational_prompt_path,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\config\prompt_operacional_ollama_v2.md
operational_prompt_sha256,94fa2a479c1995aa2cd27d9d2767e9f5bee33e56129785b393f7224b12967838


Flash,"Ver detalle{ ""type"": ""disabled"" }"
Pro,"Ver detalle{ ""type"": ""disabled"" }"
JSON_Flash,"Ver detalle{ ""type"": ""json_object"" }"
JSON_Pro,"Ver detalle{ ""type"": ""json_object"" }"
contrato_salida,"Ver detalle{ ""root_key"": ""annotations"", ""records_must_match_input_count_and_order"": true, ""record_schema"": ""LLMAnnotationPayload"" }"
caché_Flash,"Ver detalle{ ""mode"": ""automatic_prefix"", ""static_prefix_sha256"": ""d24dd948d072b83751058f7b56ea1e8294d8e6663b09c7d0596f4ec3a8acf345"", ""verified_from_usage_fields"": [ ""prompt_cache_hit_tokens"", ""prompt_cache_miss_tokens"" ] }"
caché_Pro,"Ver detalle{ ""mode"": ""automatic_prefix"", ""static_prefix_sha256"": ""d24dd948d072b83751058f7b56ea1e8294d8e6663b09c7d0596f4ec3a8acf345"", ""verified_from_usage_fields"": [ ""prompt_cache_hit_tokens"", ""prompt_cache_miss_tokens"" ] }"


Flash,"Ver detalle{ ""status"": ""credential_and_models_verified_no_corpus_sent"", ""configured_model"": ""deepseek-v4-flash"", ""model_available"": true, ""available_models"": [ ""deepseek-v4-flash"", ""deepseek-v4-pro"" ] }"
Pro,"Ver detalle{ ""status"": ""credential_and_models_verified_no_corpus_sent"", ""configured_model"": ""deepseek-v4-pro"", ""model_available"": true, ""available_models"": [ ""deepseek-v4-flash"", ""deepseek-v4-pro"" ] }"
saldo,"Ver detalle{ ""status"": ""balance_verified_no_corpus_sent"", ""is_available"": true, ""currency"": ""USD"", ""total_balance_usd"": 19.68, ""granted_balance_usd"": 0.0, ""topped_up_balance_usd"": 19.68 }"


entrada,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/processed/chunks_v2.jsonl
campaña,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/etiquetado/cascada_deepseek_v4
recuperación_histórica,Sí
checkpoint_drive_automático,No
calibración,Sí
primaria,Sí
revisión,Sí


## Carga visible del corpus

In [12]:
if globals().get('IN_COLAB'):
    from tqdm.std import tqdm  # Salida textual visible también desde VS Code.
else:
    from tqdm.auto import tqdm
from moderacion_peru.io import read_jsonl
CHUNKS=list(tqdm(read_jsonl(SOURCE),desc='Cargando chunks',unit='chunk'))
show_summary('Corpus disponible',{'chunks_totales':len(CHUNKS),'nota':'La recuperación histórica se ejecuta antes de calcular lo pendiente y su costo.'},tone='neutral')

Cargando chunks: 0chunk [00:00, ?chunk/s]

chunks_totales,166940
nota,La recuperación histórica se ejecuta antes de calcular lo pendiente y su costo.


## Funciones de ejecución, avance y checkpoint

In [13]:
from moderacion_peru.io import read_jsonl,write_json_atomic,write_jsonl_atomic
from moderacion_peru.labeling import annotate_batched_incremental,historical_recovery_signature,recover_historical_annotations
import time
if COLAB_CONTEXT is not None:
    from moderacion_peru.colab import publish_colab_outputs

def labeling_progress(description,provider):
    state={'bar':None,'last_balance_check':0.0,'balance':None,'balance_error':None,'cache_alerted':False,'low_balance_alerted':False}
    def refresh_balance(*,force=False):
        now=time.monotonic()
        if not force and now-state['last_balance_check'] < BALANCE_REFRESH_SECONDS: return
        state['last_balance_check']=now
        try:
            state['balance']=provider.balance_summary(); state['balance_error']=None
            total=state['balance']['total_balance_usd']
            if total <= LOW_BALANCE_WARNING_USD and not state['low_balance_alerted']:
                tqdm.write(f'⚠ Saldo DeepSeek bajo: US${total:.2f}. Considere recargar antes de continuar.')
                state['low_balance_alerted']=True
            elif total > LOW_BALANCE_WARNING_USD:
                state['low_balance_alerted']=False
        except Exception as exc:
            message=f'{type(exc).__name__}: {exc}'
            if message != state['balance_error']: tqdm.write(f'⚠ No se pudo actualizar el saldo DeepSeek: {message}')
            state['balance_error']=message
    def update_postfix(event):
        bar=state.get('bar'); usage=event.get('provider_usage') or {}; cache=usage.get('cache_hit_rate')
        balance=state.get('balance') or {}; total=balance.get('total_balance_usd')
        if bar is not None:
            bar.set_postfix(ok=event.get('labeled',0),errores=event.get('errors',0),gastado_USD=f"{usage.get('estimated_cost_usd',0):.4f}",saldo_USD='—' if total is None else f'{total:.2f}',caché='—' if cache is None else f'{100*cache:.1f}%')
        if usage.get('requests',0) >= CACHE_ALERT_AFTER_REQUESTS and cache is not None and cache < MIN_CACHE_HIT_RATE and not state['cache_alerted']:
            tqdm.write(f'⚠ Caché DeepSeek baja ({100*cache:.1f}%). Revise antes de ampliar la campaña; el progreso ya guardado no se pierde.')
            state['cache_alerted']=True
    def callback(event):
        if event['status']=='phase_started':
            if state.get('bar') is not None: state['bar'].close()
            label='Verificando progreso guardado' if event['phase']=='existing_progress' else 'Buscando chunks pendientes'
            state['bar']=tqdm(total=event.get('total'),desc=label,unit='chunk'); return
        if event['status']=='phase_progress':
            if state.get('bar') is not None: state['bar'].update(event.get('phase_advance',0))
            return
        if event['status']=='phase_finished':
            if state.get('bar') is not None: state['bar'].close(); state['bar']=None
            return
        if event['status']=='started':
            state['bar']=tqdm(total=event['selected'],desc=description,unit='chunk')
            refresh_balance(force=True); update_postfix(event)
            if state.get('balance') is not None and not state['balance']['is_available']:
                state['bar'].close(); state['bar']=None
                raise RuntimeError(f"DeepSeek no tiene saldo disponible (US${state['balance']['total_balance_usd']:.2f}); recargue y vuelva a ejecutar. No se envió ningún chunk pendiente.")
            return
        bar=state.get('bar')
        if bar is not None and event.get('advance'):
            bar.update(event['advance'])
            refresh_balance(); update_postfix(event)
        if event['status'] in {'finished','interrupted_checkpoint'} and bar is not None:
            refresh_balance(force=True); update_postfix(event)
            bar.close(); state['bar']=None
    return callback

def provider_run_metadata(provider,historical_recovery=None):
    probe=provider.probe()
    signature={'model':probe['model'],'thinking':probe['thinking'],'response_format':probe['response_format'],'output_contract':probe['output_contract'],'context_cache':probe['context_cache'],'prompt_sha256':probe['prompt_sha256'],'operational_prompt_sha256':probe['operational_prompt_sha256'],'records_per_request':probe['records_per_request'],'label_source':probe['label_source']}
    if historical_recovery is not None: signature['historical_recovery']=historical_recovery
    return {'provider':signature,'taxonomy':'moderacion_peru_5_salidas_v2','taxonomy_version':'2.1.0'}

FLASH_HISTORY_SIGNATURE=historical_recovery_signature(HISTORICAL_CHUNKS,HISTORICAL_FLASH_SOURCES,expected_model='deepseek-v4-flash',historical_prompt_sha256=HISTORICAL_PROMPT_SHA256) if RECOVER_HISTORICAL else None
PRO_HISTORY_SIGNATURE=historical_recovery_signature(HISTORICAL_CHUNKS,HISTORICAL_PRO_SOURCES,expected_model='deepseek-v4-pro',historical_prompt_sha256=HISTORICAL_PROMPT_SHA256) if RECOVER_HISTORICAL else None
PRIMARY_RUN_METADATA=provider_run_metadata(primary_provider,FLASH_HISTORY_SIGNATURE)
REVIEW_RUN_METADATA=provider_run_metadata(reviewer_provider,PRO_HISTORY_SIGNATURE)
if RECOVER_HISTORICAL:
    primary_recovery=recover_historical_annotations(CHUNKS,HISTORICAL_CHUNKS,HISTORICAL_FLASH_SOURCES,PRIMARY_PATH,expected_model='deepseek-v4-flash',historical_prompt_sha256=HISTORICAL_PROMPT_SHA256,run_metadata=PRIMARY_RUN_METADATA)
    review_recovery=recover_historical_annotations(CHUNKS,HISTORICAL_CHUNKS,HISTORICAL_PRO_SOURCES,REVIEW_PATH,expected_model='deepseek-v4-pro',historical_prompt_sha256=HISTORICAL_PROMPT_SHA256,run_metadata=REVIEW_RUN_METADATA,label_source='llm_remote_review_historical_recovered')
    show_result('Recuperación histórica exacta',{'Flash':primary_recovery,'Pro':review_recovery},tone='success')
    if COLAB_CONTEXT is not None and AUTO_PUBLISH_CHECKPOINTS and (primary_recovery['recovered_new'] or review_recovery['recovered_new']):
        show_result('Checkpoint histórico publicado en Drive',publish_colab_outputs(COLAB_CONTEXT),tone='success')

primary_pending=primary_recovery['pending_current_after_recovery'] if RECOVER_HISTORICAL else len(CHUNKS)
# Consumo Flash medido en el histórico por cada 5 000 chunks y tasa de caché observada de 78.56%.
scale=primary_pending/5000; input_m=8.28*scale; output_m=0.724*scale; observed_cache_rate=0.7856
cost_no_cache=input_m*0.14+output_m*0.28
cost_observed_cache=input_m*((1-observed_cache_rate)*0.14+observed_cache_rate*0.0028)+output_m*0.28
show_summary('Costo Flash de lo realmente pendiente',{'total_actual':len(CHUNKS),'recuperado_o_ya_guardado':len(CHUNKS)-primary_pending,'pendiente_Flash':primary_pending,'entrada_proyectada_M':round(input_m,2),'salida_proyectada_M':round(output_m,2),'sin_caché_USD':round(cost_no_cache,2),'con_caché_histórica_78.56%_USD':round(cost_observed_cache,2),'tope_configurado_USD':MAX_PRIMARY_COST_USD},tone='success')

def checkpoint_callback_for(output):
    checkpoint_path=output.with_suffix(output.suffix+'.checkpoint.json')
    def callback(event):
        write_json_atomic(checkpoint_path,event)
        if COLAB_CONTEXT is not None and AUTO_PUBLISH_CHECKPOINTS and event['status'] in {'periodic_checkpoint','interrupted_checkpoint'}:
            publish_colab_outputs(COLAB_CONTEXT)
    return callback

def run_campaign(rows,provider,output_name,*,limit,description):
    if not provider.probe()['credential_configured']:
        raise RuntimeError('Falta DEEPSEEK_API_KEY: configúrela como variable local o secreto privado de Colab antes de etiquetar')
    output=CAMPAIGN_ROOT/output_name
    run_metadata=PRIMARY_RUN_METADATA if output_name=='primary_flash.jsonl' else REVIEW_RUN_METADATA if output_name=='review_pro.jsonl' else provider_run_metadata(provider)
    result=annotate_batched_incremental(rows,provider,output,error_path=output.with_suffix('.errors.jsonl'),limit=limit,processing_batch_size=PROCESSING_BATCH_SIZE,progress_callback=labeling_progress(description,provider),checkpoint_callback=checkpoint_callback_for(output),checkpoint_every_batches=DRIVE_CHECKPOINT_EVERY_BATCHES,run_metadata=run_metadata,quarantine_invalid_progress=True)
    try: result['account_balance']=provider.balance_summary()
    except Exception as exc: result['account_balance_error']=f'{type(exc).__name__}: {exc}'
    write_json_atomic(output.with_suffix('.result.json'),result)
    if COLAB_CONTEXT is not None and AUTO_PUBLISH_CHECKPOINTS: publish_colab_outputs(COLAB_CONTEXT)
    return output,result

Flash,"Ver detalle{ ""schema_version"": ""1.0.0"", ""operation"": ""recover_historical_annotations"", ""signature"": { ""mapping"": ""exact_unique_video_and_normalized_text_v1"", ""expected_model"": ""deepseek-v4-flash"", ""historical_prompt_sha256"": ""52d4fec14ad433d35ec20de5f51a6954aad69dcedd1422059419dcecc2f9e778"", ""historical_chunks"": { ""name"": ""chunks_para_etiquetar.jsonl"", ""sha256"": ""eb90debf66d5e16af72c41c17c3701197e42bdcc78b81e0f914c6a49c56f8ab4"" }, ""annotations"": [ { ""name"": ""deepseek-v4-flash_labeled_chunks_seed42.jsonl"", ""sha256"": ""9ee36da617d7700a642358b53d2fb9d311fd64920bd5fc69653d5115cc1409c9"" } ] }, ""current_rows"": 166940, ""historical_rows"": 69853, ""exact_unique_matches"": 52244, ""recovered_new"": 0, ""already_present_matches"": 52244, ""ambiguous_keys_excluded"": 0, ""historical_not_reusable"": 17609, ""completed_current_after_recovery"": 52244, ""pending_current_after_recovery"": 114696, ""output_rows"": 52244, ""run_manifest"": ""D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\cascada_deepseek_v4\\primary_flash.jsonl.run.json"", ""quarantined_progress"": null, ""safety_rule"": ""only exact, unique (video_id, normalized_text) matches are reused"" }"
Pro,"Ver detalle{ ""schema_version"": ""1.0.0"", ""operation"": ""recover_historical_annotations"", ""signature"": { ""mapping"": ""exact_unique_video_and_normalized_text_v1"", ""expected_model"": ""deepseek-v4-pro"", ""historical_prompt_sha256"": ""52d4fec14ad433d35ec20de5f51a6954aad69dcedd1422059419dcecc2f9e778"", ""historical_chunks"": { ""name"": ""chunks_para_etiquetar.jsonl"", ""sha256"": ""eb90debf66d5e16af72c41c17c3701197e42bdcc78b81e0f914c6a49c56f8ab4"" }, ""annotations"": [ { ""name"": ""deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.jsonl"", ""sha256"": ""7c6c348d742610f14b0d0c15830998bb953b721a50877ff75b88c6f079d6b4af"" }, { ""name"": ""deepseek-v4-pro_revision_umbral_recalibrado_t090_seed42.jsonl"", ""sha256"": ""88370616e728d1dbd3f588bae5d3d95163b0139bfaa76586e56a6a3e672bfde2"" }, { ""name"": ""deepseek-v4-pro_revision_sospechosos_gruesos_seed42.jsonl"", ""sha256"": ""745a9dc191482fd0a8609f5ffec3266abcf7a0bd40860d7c7b541f7dac045e34"" } ] }, ""current_rows"": 166940, ""historical_rows"": 13421, ""exact_unique_matches"": 9912, ""recovered_new"": 0, ""already_present_matches"": 9912, ""ambiguous_keys_excluded"": 0, ""historical_not_reusable"": 3509, ""completed_current_after_recovery"": 9912, ""pending_current_after_recovery"": 157028, ""output_rows"": 9912, ""run_manifest"": ""D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\cascada_deepseek_v4\\review_pro.jsonl.run.json"", ""quarantined_progress"": null, ""safety_rule"": ""only exact, unique (video_id, normalized_text) matches are reused"" }"


total_actual,166940
recuperado_o_ya_guardado,52244
pendiente_Flash,114696
entrada_proyectada_M,189.94
salida_proyectada_M,16.61
sin_caché_USD,31.24
con_caché_histórica_78.56%_USD,10.77
tope_configurado_USD,60.0


## Calibración corta Flash frente a Pro

In [14]:
from moderacion_peru.labeling_calibration import select_calibration_panel,calibrate_primary_against_reviewer

PANEL_PATH=CAMPAIGN_ROOT/'calibration_panel.jsonl'
CALIBRATION_PATH=CAMPAIGN_ROOT/'calibration_flash_vs_pro.json'
if RUN_CALIBRATION:
    if PANEL_PATH.is_file():
        panel=list(tqdm(read_jsonl(PANEL_PATH),desc='Recuperando panel congelado',unit='chunk'))
        if len(panel)!=CALIBRATION_PANEL_SIZE: raise ValueError('El panel guardado no coincide con CALIBRATION_PANEL_SIZE; use otra carpeta de campaña')
    else:
        panel_progress={'bar':tqdm(total=len(CHUNKS),desc='Seleccionando panel',unit='chunk')}
        def report_panel(event):
            if event.get('advance'): panel_progress['bar'].update(event['advance'])
        panel=select_calibration_panel(CHUNKS,panel_size=CALIBRATION_PANEL_SIZE,seed=42,max_per_video=1,progress_callback=report_panel)
        panel_progress['bar'].close()
        write_jsonl_atomic(PANEL_PATH,panel)
    flash_path,flash_panel_result=run_campaign(panel,primary_provider,'calibration_flash.jsonl',limit=None,description='Calibración Flash')
    pro_path,pro_panel_result=run_campaign(panel,reviewer_provider,'calibration_pro.jsonl',limit=None,description='Calibración Pro')
    calibration=calibrate_primary_against_reviewer(read_jsonl(flash_path),read_jsonl(pro_path),minimum_auto_count=200,bootstrap_replicates=1000)
    write_json_atomic(CALIBRATION_PATH,calibration)
    show_table('Riesgo–cobertura por umbral',calibration['comparisons'],max_rows=len(calibration['comparisons']))
    show_result('Umbral operativo calibrado',calibration,tone='success' if calibration['threshold_status']=='calibrated' else 'warning')
elif CALIBRATION_PATH.is_file():
    calibration=__import__('json').loads(CALIBRATION_PATH.read_text(encoding='utf-8-sig'))
    show_table('Calibración guardada (sin repetir API)',calibration['comparisons'],max_rows=len(calibration['comparisons']))
else:
    calibration=None
    show_callout('Calibración pendiente','Active RUN_CALIBRATION=True. El panel de 1 000 es pareado por chunk y el bootstrap agrupa por video.',tone='neutral')

Seleccionando panel:   0%|          | 0/166940 [00:00<?, ?chunk/s]

Buscando chunks pendientes:   0%|          | 0/1000 [00:00<?, ?chunk/s]

Calibración Flash:   0%|          | 0/1000 [00:00<?, ?chunk/s]

⚠ Caché DeepSeek baja (22.0%). Revise antes de ampliar la campaña; el progreso ya guardado no se pierde.


Buscando chunks pendientes:   0%|          | 0/1000 [00:00<?, ?chunk/s]

Calibración Pro:   0%|          | 0/1000 [00:00<?, ?chunk/s]

⚠ Caché DeepSeek baja (0.0%). Revise antes de ampliar la campaña; el progreso ya guardado no se pierde.


threshold,auto_accepted,coverage,exact_agreement,exact_lower_one_sided_95,binary_agreement,binary_lower_one_sided_95
0.7,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.75,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.8,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.85,638,0.638,0.7288401253918495,0.6989690070394804,0.9890282131661442,0.9798859303183013
0.9,610,0.61,0.7377049180327869,0.707405834754549,0.9901639344262295,0.9810936360034461
0.95,434,0.434,0.804147465437788,0.7709696689578416,0.9976958525345622,0.9897391109328862


schema_version,1.0.0
reference_kind,stronger_llm_not_human_ground_truth
paired_chunks,1000
threshold_status,inconclusive_conservative_threshold
selected_threshold,0.95
selection_criteria,"Ver detalle{ ""minimum_exact_lower"": 0.9, ""minimum_binary_lower"": 0.95, ""minimum_auto_count"": 200 }"
comparisons,"Ver detalle[ { ""threshold"": 0.7, ""auto_accepted"": 640, ""coverage"": 0.64, ""exact_agreement"": 0.728125, ""exact_lower_one_sided_95"": 0.6982812346733331, ""binary_agreement"": 0.9890625, ""binary_lower_one_sided_95"": 0.979948410777514 }, { ""threshold"": 0.75, ""auto_accepted"": 640, ""coverage"": 0.64, ""exact_agreement"": 0.728125, ""exact_lower_one_sided_95"": 0.6982812346733331, ""binary_agreement"": 0.9890625, ""binary_lower_one_sided_95"": 0.979948410777514 }, { ""threshold"": 0.8, ""auto_accepted"": 640, ""coverage"": 0.64, ""exact_agreement"": 0.728125, ""exact_lower_one_sided_95"": 0.6982812346733331, ""binary_agreement"": 0.9890625, ""binary_lower_one_sided_95"": 0.979948410777514 }, { ""threshold"": 0.85, ""auto_accepted"": 638, ""coverage"": 0.638, ""exact_agreement"": 0.7288401253918495, ""exact_lower_one_sided_95"": 0.6989690070394804, ""binary_agreement"": 0.9890282131661442, ""binary_lower_one_sided_95"": 0.9798859303183013 }, { ""threshold"": 0.9, ""auto_accepted"": 610, ""coverage"": 0.61, ""exact_agreement"": 0.7377049180327869, ""exact_lower_one_sided_95"": 0.707405834754549, ""binary_agreement"": 0.9901639344262295, ""binary_lower_one_sided_95"": 0.9810936360034461 }, { ""threshold"": 0.95, ""auto_accepted"": 434, ""coverage"": 0.434, ""exact_agreement"": 0.804147465437788, ""exact_lower_one_sided_95"": 0.7709696689578416, ""binary_agreement"": 0.9976958525345622, ""binary_lower_one_sided_95"": 0.9897391109328862 } ]"
mean_absolute_calibration_error_exact,0.365
selected_threshold_cluster_bootstrap_95,"Ver detalle{ ""replicates"": 1000, ""exact_low"": 0.7649769585253456, ""exact_high"": 0.8410138248847926, ""binary_low"": 0.9930875576036866, ""binary_high"": 1.0 }"


## Primera pasada completa con Flash

In [ ]:
if RUN_PRIMARY:
    PRIMARY_PATH,primary_result=run_campaign(CHUNKS,primary_provider,'primary_flash.jsonl',limit=PRIMARY_LIMIT,description='Primera pasada Flash')
    show_result('Resultado Flash',primary_result,tone='success')
else:
    show_callout('Primera pasada desactivada','PRIMARY_LIMIT=None procesa todos y solo los pendientes; use 20 únicamente para un smoke mínimo. La salida reanuda por chunk_id y muestra costo, caché y saldo reales.',tone='neutral')

Verificando progreso guardado: 0chunk [00:00, ?chunk/s]

Buscando chunks pendientes:   0%|          | 0/166940 [00:00<?, ?chunk/s]

Primera pasada Flash:   0%|          | 0/114696 [00:00<?, ?chunk/s]

## Enrutamiento y revisión dirigida con Pro

In [ ]:
from moderacion_peru.labeling_calibration import build_directed_review_queue
REVIEW_QUEUE_PATH=CAMPAIGN_ROOT/'directed_review_queue.jsonl'
if RUN_DIRECTED_REVIEW:
    if calibration is None or not PRIMARY_PATH.is_file():
        raise FileNotFoundError('Complete la calibración y la primera pasada antes de revisar')
    primary_rows=list(tqdm(read_jsonl(PRIMARY_PATH),desc='Cargando propuestas Flash',unit='anotación'))
    primary_ids={row['chunk_id'] for row in primary_rows}
    paired_chunks=[row for row in tqdm(CHUNKS,desc='Uniendo chunks con Flash',unit='chunk') if row['chunk_id'] in primary_ids]
    queue_progress={'bar':tqdm(total=len(paired_chunks),desc='Construyendo cola Pro',unit='chunk')}
    def report_queue(event):
        if event.get('advance'): queue_progress['bar'].update(event['advance'])
    review_queue,routing=build_directed_review_queue(paired_chunks,primary_rows,confidence_threshold=float(calibration['selected_threshold']),safe_control_rate=SAFE_CONTROL_RATE,seed=42,progress_callback=report_queue)
    queue_progress['bar'].close()
    write_jsonl_atomic(REVIEW_QUEUE_PATH,review_queue); write_json_atomic(CAMPAIGN_ROOT/'routing_summary.json',routing)
    REVIEW_PATH,review_result=run_campaign(review_queue,reviewer_provider,'review_pro.jsonl',limit=REVIEW_LIMIT,description='Revisión dirigida Pro')
    show_summary('Enrutamiento histórico actualizado',routing,tone='success')
    show_result('Resultado Pro',review_result,tone='success')
else:
    show_callout('Revisión dirigida desactivada','Active solo después de completar Flash. Se revisan daño, abstención, baja confianza y 10% de controles seguros.',tone='neutral')

## Resultados persistidos y reportables

In [ ]:
import json
saved={}
for path in sorted(CAMPAIGN_ROOT.glob('*.result.json')):
    saved[path.stem.replace('.result','')]=json.loads(path.read_text(encoding='utf-8-sig'))
if CALIBRATION_PATH.is_file():
    current=json.loads(CALIBRATION_PATH.read_text(encoding='utf-8-sig'))
    show_table('Tabla reportable de calibración',current['comparisons'],max_rows=len(current['comparisons']))
    show_summary('Conclusión de calibración',{'estado':current['threshold_status'],'umbral':current['selected_threshold'],'pares':current['paired_chunks'],'referencia':current['reference_kind'],'bootstrap_agrupado_por_video':current['selected_threshold_cluster_bootstrap_95']},tone='success' if current['threshold_status']=='calibrated' else 'warning')
show_result('Resultados recuperados sin repetir cálculos',saved,tone='success' if saved else 'neutral')
show_callout('Límite inferencial','Flash–Pro es una calibración operativa y no reemplaza validación humana independiente. El score declarado no se interpreta como probabilidad estadística.',tone='warning')

## Publicación o checkpoint en Drive

Los archivos se generan en el SSD efímero de `/content`. Active esta celda después de un checkpoint coherente o al finalizar; publica un solo TAR.GZ y luego su manifiesto.

In [ ]:
PUBLISH_TO_DRIVE = False
if COLAB_CONTEXT is not None and PUBLISH_TO_DRIVE:
    from moderacion_peru.colab import publish_colab_outputs
    show_result('Publicación en Drive', publish_colab_outputs(COLAB_CONTEXT), tone='success')
elif COLAB_CONTEXT is not None and globals().get('AUTO_PUBLISH_CHECKPOINTS'):
    show_callout('Checkpoint automático activo', 'La recuperación, los checkpoints periódicos, Ctrl+C y cada cierre de campaña ya publican un TAR.GZ atómico en Drive.', tone='success')
elif COLAB_CONTEXT is not None:
    show_callout('Publicación desactivada', 'Cambie PUBLISH_TO_DRIVE=True tras guardar un checkpoint consistente.', tone='neutral')
else:
    show_callout('Backend local', 'Los artefactos ya permanecen en el workspace.', tone='success')

## Referencias

[1] DeepSeek, "DeepSeek V4 Preview Release," DeepSeek API Documentation, Apr. 2026. [Online]. Available: https://api-docs.deepseek.com/news/news260424/. Accessed: Aug. 5, 2026.

[2] DeepSeek, "Models and Pricing," DeepSeek API Documentation, 2026. [Online]. Available: https://api-docs.deepseek.com/quick_start/pricing. Accessed: Aug. 7, 2026.

[3] B. Settles, "Active Learning Literature Survey," Univ. Wisconsin–Madison, Computer Sciences Tech. Rep. 1648, 2009. [Online]. Available: https://minds.wisconsin.edu/handle/1793/60660

[4] H. Schroeder, D. Roy, and J. Kabbara, "Just Put a Human in the Loop? Investigating LLM-Assisted Annotation for Subjective Tasks," in Findings ACL, 2025, pp. 25771–25795, doi: 10.18653/v1/2025.findings-acl.1323.

[5] Google Colab, "Known Issues and Workarounds," googlecolab/colab-vscode Wiki, 2026. [Online]. Available: https://github.com/googlecolab/colab-vscode/wiki/Known-Issues-and-Workarounds. Accessed: Aug. 5, 2026.

[6] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.